In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost
import sklearn

I have decided to work on the hourly dataset :

In [29]:
df=pd.read_csv('/content/hour.csv')

In [30]:
df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [31]:
len(df)

17379

In [32]:
df.drop('instant',axis=1,inplace=True)

In [33]:
df['dteday']=pd.to_datetime(df['dteday'])

Let's create some features :

Cyclical temporical features :

In [34]:
df['hr_sin']=np.sin(2*np.pi*(df['hr']/24))
df['hr_cos']=np.cos(2*np.pi*(df['hr']/24))
df['mnth_sin']=np.sin(2*np.pi*(df['mnth']/12))
df['mnth_cos']=np.cos(2*np.pi*(df['mnth']/12))
df['week_sin']=np.sin(2*np.pi*(df['weekday']/7))
df['week_cos']=np.cos(2*np.pi*(df['weekday']/7))

Commute features :

In [35]:
df['Rush_Hour']=((df['workingday']==1)&(df['hr'].isin([7,8,9,17,18,19]))).astype(bool)
df['Off_Peak']=(df['hr'].isin([0,1,2,3,4,5])).astype(bool)

In [36]:
df['Previous_hour_cnt']=df['cnt'].shift(1)
df['Previous_day_cnt']=df['cnt'].shift(24)

We create now weather based columns :

In [37]:
df['delta_temp']=df['temp']-df['atemp']
df['temp*hum']=df['temp']*df['hum']
df['bad_weather']=(df['weathersit']>=3).astype(bool)
df['windspeed*temp']=(df['temp']*df['windspeed'])

Let's create rolling averages feature:

In [38]:
df['cnt_roll_3h_mean']=(df['cnt'].shift(1).rolling(window=3).mean())
df['cnt_roll_3h_std']=(df['cnt'].shift(1).rolling(window=3).std())
df['cnt_roll_24_std']=(df['cnt'].shift(1).rolling(window=24).std())
df['cnt_roll_same_hour_7d_mean']=(df.groupby('hr')['cnt'].transform(lambda x:x.shift(1).rolling(window=7).mean()))

In [39]:
#We log transform target since it is righly skewed:
df['log_cnt']=np.log1p(df['cnt'])
df.drop('cnt',axis=1,inplace=True)

In [40]:
df.dropna(inplace=True)

In [41]:
len(df)*0.8

13768.800000000001

In [42]:
X=df.drop(['log_cnt','casual','registered','dteday'],axis=1)
y=df['log_cnt']
X_train=X[:13768]
X_test=X[13768:]
y_train=y[:13768]
y_test=y[13768:]

In [43]:
from sklearn.ensemble import RandomForestRegressor
v0=RandomForestRegressor()
v0.fit(X_train,y_train)

RandomForestRegressor()

In [44]:
from sklearn.metrics import mean_absolute_error
y_pred_0=v0.predict(X_test)
mae_0=mean_absolute_error(y_test,y_pred_0 )
(mae_0 /y_test.median())*100

3.335995422344796

Our first model is performing well let's try to improve things further with RandomizedSearchCV

In [45]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV,TimeSeriesSplit


tscv=TimeSeriesSplit(n_splits=5)
xgb_model = XGBRegressor(
    tree_method="hist",
    device="cuda",
    random_state=42
)

xgb_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [6, 8, 10,12],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree':[0.6,0.8,1.0],
    'gamma':[0,0.1,0.2]
}

randomized_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_grid,
    n_iter=40,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    verbose=1
)

randomized_search.fit(X_train, y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [14:55:21] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


RandomizedSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device='cuda',
                                          early_stopping_rounds=None,
                                          enable_categorical=True,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gam...
                                          monotone_constraints=None,
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None, ...),
                   n_iter=40,
                   param_distributions={'colsample_bytree': [0.6, 0.8, 1.0],
                                        'gamma': [0, 0.1, 0.2],
                                        'learning_rate': [0.01, 0.05, 0.1],
                                        'max_depth': [6, 8, 10, 12],
                                        'n_estimators': [100, 300, 500],
                                        'subsample': [0.6, 0.8, 1.0]},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=1)

In [46]:
print(randomized_search.best_params_)
#{'subsample': 0.8, 'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.05, 'gamma': 0.2, 'colsample_bytree': 0.6}

{'subsample': 0.8, 'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.05, 'gamma': 0.2, 'colsample_bytree': 0.6}


In [47]:
v1=XGBRegressor(**randomized_search.best_params_,random_state=42,tree_method='hist',device='cuda')
v1.fit(X_train,y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device='cuda', early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0.2, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [48]:
from sklearn.metrics import root_mean_squared_error
y_pred_1=v1.predict(X_test)
rmse_1=root_mean_squared_error(np.expm1(y_test),np.expm1(y_pred_1))
rmse_1

47.17170327963064